# Machine Learning Basics for QSAR

This notebook accompanies the QSAR tutorial chapter: **Machine Learning Basics for QSAR**.

## Prepare descriptor data

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

from rdkit import Chem
from rdkit.Chem import Descriptors

def find_example_dataset():
    candidates = [
        Path("../data/example_qsar_dataset.csv"),
        Path("data/example_qsar_dataset.csv"),
        Path("../../data/example_qsar_dataset.csv"),
    ]
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError("Could not find example_qsar_dataset.csv.")

def mol_from_smiles(smiles):
    try:
        return Chem.MolFromSmiles(smiles)
    except Exception:
        return None

def calculate_all_rdkit_descriptors(mol):
    return Descriptors.CalcMolDescriptors(mol, missingVal=np.nan)

# Load data
df = pd.read_csv(find_example_dataset())

# Convert SMILES to RDKit molecules and remove invalid molecules
df["Mol"] = df["SMILES"].apply(mol_from_smiles)
df_clean = df.dropna(subset=["Mol"]).copy()

# Calculate all default RDKit descriptors
descriptor_rows = [
    calculate_all_rdkit_descriptors(mol)
    for mol in df_clean["Mol"]
]

X = pd.DataFrame(descriptor_rows, index=df_clean.index)

# Convert to numeric and remove descriptor columns containing missing values
X = X.apply(pd.to_numeric, errors="coerce")
X = X.dropna(axis=1)

# Target values
y = df_clean["Activity"].astype(float)

print("Number of valid molecules:", len(df_clean))
print("Number of RDKit descriptors:", X.shape[1])

X.head()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import VarianceThreshold
import numpy as np
import pandas as pd

# Remove constant and near-constant descriptors
selector = VarianceThreshold(threshold=1e-8)
X_var = pd.DataFrame(
    selector.fit_transform(X),
    columns=X.columns[selector.get_support()],
    index=X.index
)

X_train, X_test, y_train, y_test = train_test_split(
    X_var,
    y,
    test_size=0.2,
    random_state=42
)

# Reduce to a manageable descriptor set for beginner feature-selection examples
# Use descriptors with highest variance.
top_variance_features = X_train.var().sort_values(ascending=False).head(30).index
X_train_small = X_train[top_variance_features]
X_test_small = X_test[top_variance_features]

print("Using", X_train_small.shape[1], "candidate descriptors.")

## What is a model? Train a simple Random Forest regression model

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np

model = RandomForestRegressor(
    n_estimators=300,
    random_state=42
)

model.fit(X_train_small, y_train)

y_pred_train = model.predict(X_train_small)
y_pred_test = model.predict(X_test_small)

print("Training R2:", r2_score(y_train, y_pred_train))
print("Test R2:", r2_score(y_test, y_pred_test))
print("Test MAE:", mean_absolute_error(y_test, y_pred_test))
print("Test RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_test)))

## Regression vs. classification

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Binary class based on the example dataset
y_class = df_clean.loc[y.index, "Class"].astype(int)

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_var,
    y_class,
    test_size=0.2,
    random_state=42,
    stratify=y_class
)

X_train_c = X_train_c[top_variance_features]
X_test_c = X_test_c[top_variance_features]

clf = RandomForestClassifier(
    n_estimators=300,
    random_state=42
)

clf.fit(X_train_c, y_train_c)
y_pred_c = clf.predict(X_test_c)

print("Accuracy:", accuracy_score(y_test_c, y_pred_c))
print("Confusion matrix:")
print(confusion_matrix(y_test_c, y_pred_c))
print(classification_report(y_test_c, y_pred_c))

## Overfitting and underfitting demonstration

In [ ]:
from sklearn.tree import DecisionTreeRegressor

shallow_tree = DecisionTreeRegressor(max_depth=2, random_state=42)
deep_tree = DecisionTreeRegressor(max_depth=None, random_state=42)

shallow_tree.fit(X_train_small, y_train)
deep_tree.fit(X_train_small, y_train)

for name, fitted_model in [
    ("Shallow tree", shallow_tree),
    ("Deep tree", deep_tree)
]:
    train_pred = fitted_model.predict(X_train_small)
    test_pred = fitted_model.predict(X_test_small)

    print("\n" + name)
    print("Training R2:", r2_score(y_train, train_pred))
    print("Test R2:", r2_score(y_test, test_pred))

## Sequential feature selection

In [ ]:
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold

estimator = RandomForestRegressor(
    n_estimators=150,
    random_state=42
)

cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

sfs = SequentialFeatureSelector(
    estimator=estimator,
    n_features_to_select=8,
    direction="forward",
    scoring="r2",
    cv=cv,
    n_jobs=-1
)

sfs.fit(X_train_small, y_train)

selected_features_sfs = X_train_small.columns[sfs.get_support()]

print("Sequentially selected descriptors:")
print(selected_features_sfs.tolist())

## Simple genetic algorithm for feature selection

In [ ]:
import numpy as np
from sklearn.model_selection import cross_val_score, KFold
from sklearn.ensemble import RandomForestRegressor

rng = np.random.default_rng(42)

candidate_features = list(X_train_small.columns)
n_features_total = len(candidate_features)
n_features_to_select = 8

population_size = 20
n_generations = 10
mutation_rate = 0.15

cv = KFold(n_splits=5, shuffle=True, random_state=42)

estimator = RandomForestRegressor(
    n_estimators=120,
    random_state=42
)

def random_individual():
    individual = np.zeros(n_features_total, dtype=bool)
    selected = rng.choice(
        n_features_total,
        size=n_features_to_select,
        replace=False
    )
    individual[selected] = True
    return individual

def repair_individual(individual):
    individual = individual.copy()

    # Ensure exactly n_features_to_select features are selected
    while individual.sum() > n_features_to_select:
        selected_positions = np.where(individual)[0]
        individual[rng.choice(selected_positions)] = False

    while individual.sum() < n_features_to_select:
        unselected_positions = np.where(~individual)[0]
        individual[rng.choice(unselected_positions)] = True

    return individual

def fitness(individual):
    selected = np.array(candidate_features)[individual]
    scores = cross_val_score(
        estimator,
        X_train_small[selected],
        y_train,
        cv=cv,
        scoring="r2"
    )
    return scores.mean()

# Initial population
population = [random_individual() for _ in range(population_size)]

best_individual = None
best_score = -np.inf

for generation in range(n_generations):
    scores = np.array([fitness(ind) for ind in population])

    if scores.max() > best_score:
        best_score = scores.max()
        best_individual = population[scores.argmax()].copy()

    # Select top half as parents
    parent_indices = scores.argsort()[-population_size // 2:]
    parents = [population[i] for i in parent_indices]

    # Create next generation
    next_population = parents.copy()

    while len(next_population) < population_size:
        p1, p2 = rng.choice(len(parents), size=2, replace=False)
        parent1 = parents[p1]
        parent2 = parents[p2]

        # Crossover
        crossover_mask = rng.random(n_features_total) < 0.5
        child = np.where(crossover_mask, parent1, parent2)

        # Mutation
        mutation_mask = rng.random(n_features_total) < mutation_rate
        child = np.logical_xor(child, mutation_mask)

        child = repair_individual(child)
        next_population.append(child)

    population = next_population

    print(f"Generation {generation + 1}: best CV R2 = {best_score:.3f}")

selected_features_ga = np.array(candidate_features)[best_individual]

print("\nBest GA CV R2:", best_score)
print("GA-selected descriptors:")
print(selected_features_ga.tolist())

In [ ]:
# Train final model using GA-selected descriptors and evaluate on external test set
final_model = RandomForestRegressor(
    n_estimators=300,
    random_state=42
)

final_model.fit(X_train_small[selected_features_ga], y_train)

y_pred_test = final_model.predict(X_test_small[selected_features_ga])

print("External test R2:", r2_score(y_test, y_pred_test))
print("External test MAE:", mean_absolute_error(y_test, y_pred_test))
print("External test RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_test)))